# Team Deep Dive Dashboard

Interactive analysis for a specific team/manager.

In [ ]:
spark.sql("USE CATALOG workspace")

## Select Your Team

In [ ]:
# Get all managers in League of Inches
managers_df = spark.sql("""
SELECT DISTINCT manager_display_name
FROM workspace.sleeper_core.dim_manager_roster_map
WHERE cluster_key = 'league_of_inches'
ORDER BY manager_display_name
""")

managers = [row.manager_display_name for row in managers_df.collect()]
print("Available managers:", managers)

In [ ]:
# Create dropdown widget
dbutils.widgets.dropdown("selected_manager", managers[0], managers, "Select Manager")

# Get selected value
selected_manager = dbutils.widgets.get("selected_manager")
print(f"Analyzing team: {selected_manager}")

---

# Team Overview

## 1. Top Scorers (All-Time)

Best players who ever started for this team.

In [ ]:
top_scorers = spark.sql(f"""
WITH player_games_started AS (
  SELECT
    mr.manager_display_name,
    pw.player_id,
    COUNT(*) AS games_started
  FROM workspace.sleeper_core.fact_player_week pw
  LEFT JOIN workspace.sleeper_core.dim_manager_roster_map mr
    ON pw.league_id = mr.league_id
    AND pw.roster_id = mr.roster_id
    AND pw.season = mr.season
  WHERE mr.cluster_key = 'league_of_inches'
    AND mr.manager_display_name = '{selected_manager}'
    AND pw.was_started = TRUE
  GROUP BY mr.manager_display_name, pw.player_id
),
player_totals AS (
  SELECT
    pt.player_id,
    MAX(pt.full_name) AS player_name,
    MAX(pt.position) AS position,
    SUM(pt.points_as_starter) AS total_points,
    MIN(pt.first_season) AS first_season,
    MAX(pt.last_season) AS last_season
  FROM workspace.sleeper_core.agg_player_roster_totals pt
  LEFT JOIN workspace.sleeper_core.dim_manager_roster_map mr
    ON pt.league_id = mr.league_id
    AND pt.roster_id = mr.roster_id
  WHERE mr.cluster_key = 'league_of_inches'
    AND mr.manager_display_name = '{selected_manager}'
  GROUP BY pt.player_id
)
SELECT
  pt.player_name,
  pt.position,
  ROUND(pt.total_points, 1) AS points_when_started,
  gs.games_started,
  ROUND(pt.total_points / gs.games_started, 2) AS ppg_when_started,
  pt.first_season,
  pt.last_season
FROM player_totals pt
LEFT JOIN player_games_started gs ON pt.player_id = gs.player_id
ORDER BY pt.total_points DESC
LIMIT 20
""")

display(top_scorers)

## 2. Top Scorers (Non-QB)

In [ ]:
top_scorers_no_qb = spark.sql(f"""
WITH player_games_started AS (
  SELECT
    mr.manager_display_name,
    pw.player_id,
    COUNT(*) AS games_started
  FROM workspace.sleeper_core.fact_player_week pw
  LEFT JOIN workspace.sleeper_core.dim_manager_roster_map mr
    ON pw.league_id = mr.league_id
    AND pw.roster_id = mr.roster_id
    AND pw.season = mr.season
  LEFT JOIN workspace.sleeper_core.dim_players p ON pw.player_id = p.player_id
  WHERE mr.cluster_key = 'league_of_inches'
    AND mr.manager_display_name = '{selected_manager}'
    AND pw.was_started = TRUE
    AND p.position != 'QB'
  GROUP BY mr.manager_display_name, pw.player_id
),
player_totals AS (
  SELECT
    pt.player_id,
    MAX(pt.full_name) AS player_name,
    MAX(pt.position) AS position,
    SUM(pt.points_as_starter) AS total_points,
    MIN(pt.first_season) AS first_season,
    MAX(pt.last_season) AS last_season
  FROM workspace.sleeper_core.agg_player_roster_totals pt
  LEFT JOIN workspace.sleeper_core.dim_manager_roster_map mr
    ON pt.league_id = mr.league_id
    AND pt.roster_id = mr.roster_id
  WHERE mr.cluster_key = 'league_of_inches'
    AND mr.manager_display_name = '{selected_manager}'
    AND pt.position != 'QB'
  GROUP BY pt.player_id
)
SELECT
  pt.player_name,
  pt.position,
  ROUND(pt.total_points, 1) AS points_when_started,
  gs.games_started,
  ROUND(pt.total_points / gs.games_started, 2) AS ppg_when_started,
  pt.first_season,
  pt.last_season
FROM player_totals pt
LEFT JOIN player_games_started gs ON pt.player_id = gs.player_id
ORDER BY pt.total_points DESC
LIMIT 20
""")

display(top_scorers_no_qb)

## 3. Draft Picks

All players this team has drafted and their career performance.

In [ ]:
draft_picks = spark.sql(f"""
WITH draft_picks AS (
  SELECT
    dr.season,
    CAST(dp.round AS INT) AS round,
    dp.pick_no,
    dp.roster_id,
    dp.player_id,
    lower(regexp_replace(coalesce(li.name, 'unknown'), '[^a-zA-Z0-9]+', '_')) AS cluster_key
  FROM workspace.sleeper_raw.sleeper_draft_picks_snapshot dp
  JOIN workspace.sleeper_raw.sleeper_drafts_snapshot dr ON dp.draft_id = dr.draft_id
  JOIN workspace.sleeper_raw.sleeper_league_info_snapshot li ON dr.league_id = li.league_id
  WHERE dp.player_id IS NOT NULL
),
career_points AS (
  SELECT
    p.season AS draft_season,
    p.round,
    p.pick_no,
    p.player_id,
    pl.full_name,
    pl.position,
    COALESCE(SUM(pw.points), 0) AS total_career_points,
    COUNT(DISTINCT pw.season) AS seasons_played,
    COUNT(*) AS games_played
  FROM draft_picks p
  LEFT JOIN workspace.sleeper_core.dim_players pl ON p.player_id = pl.player_id
  LEFT JOIN workspace.sleeper_core.fact_player_week_enriched pw
    ON p.cluster_key = pw.cluster_key
    AND p.player_id = pw.player_id
    AND CAST(pw.season AS INT) >= CAST(p.season AS INT)
  LEFT JOIN workspace.sleeper_core.dim_manager_roster_map mr
    ON p.roster_id = mr.roster_id
    AND p.cluster_key = mr.cluster_key
    AND p.season = mr.season
  WHERE p.cluster_key = 'league_of_inches'
    AND mr.manager_display_name = '{selected_manager}'
  GROUP BY p.season, p.round, p.pick_no, p.player_id, pl.full_name, pl.position
),
points_for_manager AS (
  SELECT
    pw.player_id,
    SUM(CASE WHEN pw.was_started THEN pw.points ELSE 0 END) AS started_points_for_you
  FROM workspace.sleeper_core.fact_player_week pw
  LEFT JOIN workspace.sleeper_core.dim_manager_roster_map mr
    ON pw.league_id = mr.league_id
    AND pw.roster_id = mr.roster_id
    AND pw.season = mr.season
  WHERE mr.cluster_key = 'league_of_inches'
    AND mr.manager_display_name = '{selected_manager}'
  GROUP BY pw.player_id
)
SELECT
  cp.draft_season,
  CONCAT('Round ', cp.round, ' Pick ', cp.pick_no) AS pick_description,
  cp.full_name,
  cp.position,
  ROUND(cp.total_career_points, 1) AS career_points,
  ROUND(COALESCE(pfm.started_points_for_you, 0), 1) AS started_for_you,
  cp.games_played,
  cp.seasons_played
FROM career_points cp
LEFT JOIN points_for_manager pfm ON cp.player_id = pfm.player_id
ORDER BY cp.draft_season DESC, cp.round ASC, cp.pick_no ASC
""")

display(draft_picks)

## 4. Best Draft Picks

In [ ]:
best_draft_picks = spark.sql(f"""
WITH draft_picks AS (
  SELECT
    dr.season,
    CAST(dp.round AS INT) AS round,
    dp.pick_no,
    dp.roster_id,
    dp.player_id,
    lower(regexp_replace(coalesce(li.name, 'unknown'), '[^a-zA-Z0-9]+', '_')) AS cluster_key
  FROM workspace.sleeper_raw.sleeper_draft_picks_snapshot dp
  JOIN workspace.sleeper_raw.sleeper_drafts_snapshot dr ON dp.draft_id = dr.draft_id
  JOIN workspace.sleeper_raw.sleeper_league_info_snapshot li ON dr.league_id = li.league_id
  WHERE dp.player_id IS NOT NULL
),
career_points AS (
  SELECT
    p.season AS draft_season,
    p.round,
    p.pick_no,
    p.player_id,
    pl.full_name,
    pl.position,
    COALESCE(SUM(pw.points), 0) AS total_career_points,
    COUNT(DISTINCT pw.season) AS seasons_played,
    COUNT(*) AS games_played
  FROM draft_picks p
  LEFT JOIN workspace.sleeper_core.dim_players pl ON p.player_id = pl.player_id
  LEFT JOIN workspace.sleeper_core.fact_player_week_enriched pw
    ON p.cluster_key = pw.cluster_key
    AND p.player_id = pw.player_id
    AND CAST(pw.season AS INT) >= CAST(p.season AS INT)
  LEFT JOIN workspace.sleeper_core.dim_manager_roster_map mr
    ON p.roster_id = mr.roster_id
    AND p.cluster_key = mr.cluster_key
    AND p.season = mr.season
  WHERE p.cluster_key = 'league_of_inches'
    AND mr.manager_display_name = '{selected_manager}'
  GROUP BY p.season, p.round, p.pick_no, p.player_id, pl.full_name, pl.position
),
points_for_manager AS (
  SELECT
    pw.player_id,
    SUM(CASE WHEN pw.was_started THEN pw.points ELSE 0 END) AS started_points_for_you
  FROM workspace.sleeper_core.fact_player_week pw
  LEFT JOIN workspace.sleeper_core.dim_manager_roster_map mr
    ON pw.league_id = mr.league_id
    AND pw.roster_id = mr.roster_id
    AND pw.season = mr.season
  WHERE mr.cluster_key = 'league_of_inches'
    AND mr.manager_display_name = '{selected_manager}'
  GROUP BY pw.player_id
)
SELECT
  cp.full_name,
  cp.position,
  cp.draft_season,
  CONCAT('Round ', cp.round, ' Pick ', cp.pick_no) AS pick_description,
  ROUND(cp.total_career_points, 1) AS career_points,
  ROUND(COALESCE(pfm.started_points_for_you, 0), 1) AS started_for_you,
  cp.games_played,
  cp.seasons_played
FROM career_points cp
LEFT JOIN points_for_manager pfm ON cp.player_id = pfm.player_id
WHERE cp.games_played > 0
ORDER BY cp.total_career_points DESC
LIMIT 10
""")

display(best_draft_picks)

## 5. Worst Draft Picks (Busts)

In [ ]:
worst_draft_picks = spark.sql(f"""
WITH draft_picks AS (
  SELECT
    dr.season,
    CAST(dp.round AS INT) AS round,
    dp.pick_no,
    dp.roster_id,
    dp.player_id,
    lower(regexp_replace(coalesce(li.name, 'unknown'), '[^a-zA-Z0-9]+', '_')) AS cluster_key
  FROM workspace.sleeper_raw.sleeper_draft_picks_snapshot dp
  JOIN workspace.sleeper_raw.sleeper_drafts_snapshot dr ON dp.draft_id = dr.draft_id
  JOIN workspace.sleeper_raw.sleeper_league_info_snapshot li ON dr.league_id = li.league_id
  WHERE dp.player_id IS NOT NULL
),
career_points AS (
  SELECT
    p.season AS draft_season,
    p.round,
    p.pick_no,
    p.player_id,
    pl.full_name,
    pl.position,
    COALESCE(SUM(pw.points), 0) AS total_career_points,
    COUNT(DISTINCT pw.season) AS seasons_played,
    COUNT(*) AS games_played
  FROM draft_picks p
  LEFT JOIN workspace.sleeper_core.dim_players pl ON p.player_id = pl.player_id
  LEFT JOIN workspace.sleeper_core.fact_player_week_enriched pw
    ON p.cluster_key = pw.cluster_key
    AND p.player_id = pw.player_id
    AND CAST(pw.season AS INT) >= CAST(p.season AS INT)
  LEFT JOIN workspace.sleeper_core.dim_manager_roster_map mr
    ON p.roster_id = mr.roster_id
    AND p.cluster_key = mr.cluster_key
    AND p.season = mr.season
  WHERE p.cluster_key = 'league_of_inches'
    AND mr.manager_display_name = '{selected_manager}'
  GROUP BY p.season, p.round, p.pick_no, p.player_id, pl.full_name, pl.position
),
points_for_manager AS (
  SELECT
    pw.player_id,
    SUM(CASE WHEN pw.was_started THEN pw.points ELSE 0 END) AS started_points_for_you
  FROM workspace.sleeper_core.fact_player_week pw
  LEFT JOIN workspace.sleeper_core.dim_manager_roster_map mr
    ON pw.league_id = mr.league_id
    AND pw.roster_id = mr.roster_id
    AND pw.season = mr.season
  WHERE mr.cluster_key = 'league_of_inches'
    AND mr.manager_display_name = '{selected_manager}'
  GROUP BY pw.player_id
)
SELECT
  cp.full_name,
  cp.position,
  cp.draft_season,
  CONCAT('Round ', cp.round, ' Pick ', cp.pick_no) AS pick_description,
  ROUND(cp.total_career_points, 1) AS career_points,
  ROUND(COALESCE(pfm.started_points_for_you, 0), 1) AS started_for_you,
  cp.games_played,
  cp.seasons_played
FROM career_points cp
LEFT JOIN points_for_manager pfm ON cp.player_id = pfm.player_id
ORDER BY cp.total_career_points ASC, cp.round ASC
LIMIT 10
""")

display(worst_draft_picks)

## 6. Roster Churn (Boomerang Players)

Players this team dropped and re-acquired multiple times.

In [ ]:
boomerang_players = spark.sql(f"""
WITH manager_ownership AS (
  SELECT
    po.player_id,
    p.full_name,
    p.position,
    COUNT(DISTINCT po.ownership_id) AS times_acquired,
    -- True departures = times acquired minus currently active ownerships
    COUNT(DISTINCT po.ownership_id) - SUM(CASE WHEN po.is_current_roster THEN 1 ELSE 0 END) AS times_departed,
    MIN(po.season) AS first_season,
    MAX(po.season) AS last_season,
    MAX(CASE WHEN po.is_current_roster THEN 'Yes' ELSE 'No' END) AS currently_rostered
  FROM workspace.sleeper_core.dim_player_ownership po
  LEFT JOIN workspace.sleeper_core.dim_players p ON po.player_id = p.player_id
  LEFT JOIN workspace.sleeper_core.dim_manager_roster_map mr
    ON po.league_id = mr.league_id
    AND po.roster_id = mr.roster_id
    AND po.season = mr.season
  WHERE mr.cluster_key = 'league_of_inches'
    AND mr.manager_display_name = '{selected_manager}'
  GROUP BY po.player_id, p.full_name, p.position
  HAVING COUNT(DISTINCT po.ownership_id) > 1
)
SELECT
  full_name,
  position,
  times_acquired,
  times_departed,
  first_season,
  last_season,
  currently_rostered
FROM manager_ownership
ORDER BY times_acquired DESC, times_departed DESC
LIMIT 20
""")

display(boomerang_players)

## 7. Trade History

All trades this team has made and their outcomes.

In [ ]:
trades = spark.sql(f"""
SELECT
  trade_season,
  CASE 
    WHEN roster_a_manager = '{selected_manager}' THEN roster_b_manager
    ELSE roster_a_manager
  END AS traded_with,
  CASE
    WHEN roster_a_manager = '{selected_manager}' THEN ROUND(roster_a_points, 1)
    ELSE ROUND(roster_b_points, 1)
  END AS my_points,
  CASE
    WHEN roster_a_manager = '{selected_manager}' THEN ROUND(roster_b_points, 1)
    ELSE ROUND(roster_a_points, 1)
  END AS their_points,
  CASE
    WHEN winner_manager = '{selected_manager}' THEN 'Won'
    WHEN loser_manager = '{selected_manager}' THEN 'Lost'
    ELSE 'Tie'
  END AS outcome,
  ROUND(trade_impact_magnitude, 1) AS point_differential,
  CASE WHEN trade_is_complete THEN 'Complete' ELSE 'Pending' END AS status
FROM workspace.sleeper_trades.agg_trade_winners_enriched
WHERE cluster_name = 'League of Inches'
  AND (roster_a_manager = '{selected_manager}' OR roster_b_manager = '{selected_manager}')
ORDER BY trade_season DESC, point_differential DESC
""")

display(trades)

## 8. Start/Sit Decisions

Players with biggest difference between bench and starting performance.

In [ ]:
start_sit = spark.sql(f"""
WITH player_splits AS (
  SELECT
    pw.player_id,
    p.full_name,
    p.position,
    COUNT(CASE WHEN pw.was_started THEN 1 END) AS games_started,
    COALESCE(AVG(CASE WHEN pw.was_started THEN pw.points END), 0) AS ppg_started,
    COUNT(CASE WHEN NOT pw.was_started THEN 1 END) AS games_benched,
    COALESCE(AVG(CASE WHEN NOT pw.was_started THEN pw.points END), 0) AS ppg_benched
  FROM workspace.sleeper_core.fact_player_week pw
  LEFT JOIN workspace.sleeper_core.dim_players p ON pw.player_id = p.player_id
  LEFT JOIN workspace.sleeper_core.dim_manager_roster_map mr
    ON pw.league_id = mr.league_id
    AND pw.roster_id = mr.roster_id
    AND pw.season = mr.season
  WHERE mr.cluster_key = 'league_of_inches'
    AND mr.manager_display_name = '{selected_manager}'
  GROUP BY pw.player_id, p.full_name, p.position
  HAVING COUNT(*) >= 5
)
SELECT
  full_name,
  position,
  games_started,
  ROUND(ppg_started, 2) AS ppg_started,
  games_benched,
  ROUND(ppg_benched, 2) AS ppg_benched,
  ROUND(ppg_benched - ppg_started, 2) AS bench_advantage
FROM player_splits
WHERE games_benched >= 3 AND games_started >= 3
ORDER BY bench_advantage DESC
LIMIT 20
""")

display(start_sit)

---

## Summary

This dashboard shows:
1. ✅ Top scorers (all-time career)
2. ✅ Top scorers (non-QB)
3. ✅ All draft picks with career points
4. ✅ Best draft picks (hits)
5. ✅ Worst draft picks (busts)
6. ✅ Roster churn (boomerang players)
7. ✅ Trade history and outcomes
8. ✅ Start/sit decision analysis

**To switch teams**: Use the dropdown at the top and re-run all cells.